# 🎙️ F5-TTS Demo — Cloud Notebook (3 models)

Run this in **Google Colab** (free T4 GPU) or **Kaggle** (free T4×2). No local install needed.

| Tag         | Source                                                          | Size    |
|-------------|-----------------------------------------------------------------|---------|
| `base`      | `SWivid/F5-TTS` (`F5TTS_v1_Base`) — auto-downloaded             | ~1.2 GB |
| `f5_indo`   | [`VesperGrovemane/f5_indo`](https://huggingface.co/VesperGrovemane/f5_indo) → `f5_indo/f5_indo.pt`    | 1.35 GB |
| `f5_indocom`| [`VesperGrovemane/f5_indo`](https://huggingface.co/VesperGrovemane/f5_indo) → `f5_indocom/f5_indocom.pt` | 5.4 GB |

**How to use:**
1. Run cells **1 → 6** once (installs, downloads, sets up)
2. Run cell **7** (or 8, or 9) to load **just the model you want**
3. Run cell **10** to type a sentence and hear all 3 models at once

GPU runtime recommended: **Runtime → Change runtime type → T4 GPU** (Colab).

## 1. Install dependencies

Only run this once. ~2 minutes.

In [ ]:
!pip install -q f5_tts huggingface_hub soundfile 2>&1 | tail -5
print('✓ install done')

## 2. Imports & config

In [ ]:
import os, sys, warnings, shutil, tempfile, time
from pathlib import Path
warnings.filterwarnings("ignore")

import torch, torchaudio, numpy as np
from IPython.display import Audio, display, HTML

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

CKPTS_DIR   = Path("ckpts")
OUTPUT_DIR  = Path("outputs")
CKPTS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

HF_REPO_ID = "VesperGrovemane/f5_indo"
MODEL_CFG  = dict(dim=1024, depth=22, heads=16, ff_mult=2,
                  conv_layers=4, text_dim=512)
MEL_SPEC    = "vocos"
SAMPLE_RATE = 24000

## 3. Reference audio

Upload your own (recommended), or use the bundled one.
5–15 seconds of clean speech works best. We'll auto-resample to 24 kHz mono.

In [ ]:
REF_AUDIO = None  # will be set below
REF_TEXT  = "saya membeli tiga buah apel di pasar."  # from HF repo's audio_2_transcript.txt

# ------------------------------------------------------------------
# Resolve the reference audio: try HF → local file → manual upload
# ------------------------------------------------------------------
import urllib.request

HF_REF_AUDIO = f"https://huggingface.co/{HF_REPO_ID}/resolve/main/ref_audio/audio_2.wav"
HF_REF_TRANSCRIPT = f"https://huggingface.co/{HF_REPO_ID}/resolve/main/ref_audio/audio_2_transcript.txt"
LOCAL_REF_AUDIO = Path("ref_audio/audio_2.wav")
WORK_REF_AUDIO = Path("ref_audio/audio_2.wav")  # where we'll save it for this session
WORK_REF_AUDIO.parent.mkdir(parents=True, exist_ok=True)

# 1) Try HuggingFace first (works on Colab/Kaggle/local alike)
if not WORK_REF_AUDIO.exists():
    print(f"⏳ downloading reference audio from HF ...")
    print(f"   {HF_REF_AUDIO}")
    try:
        urllib.request.urlretrieve(HF_REF_AUDIO, WORK_REF_AUDIO)
        print(f"   ✓ downloaded  ({os.path.getsize(WORK_REF_AUDIO)/1024:.1f} KB)")
        # Also pull the transcript (in case the user updated it on HF)
        try:
            with urllib.request.urlopen(HF_REF_TRANSCRIPT, timeout=10) as r:
                REF_TEXT = r.read().decode("utf-8").strip()
            print(f"   ✓ transcript from HF: {REF_TEXT!r}")
        except Exception as e:
            print(f"   (transcript fetch failed: {e}, using default)")
    except Exception as e:
        print(f"   HF download failed: {e}")

# 2) If still missing, look in the local working dir (user dropped a file)
if not WORK_REF_AUDIO.exists() and LOCAL_REF_AUDIO.exists():
    shutil.copy(LOCAL_REF_AUDIO, WORK_REF_AUDIO)
    print(f"✓ using local file: {LOCAL_REF_AUDIO}")

# 3) Last resort: ask the user to upload (Colab / Kaggle)
if not WORK_REF_AUDIO.exists():
    try:
        from google.colab import files
        print("📤 Upload your reference audio (any sample rate, mono or stereo):")
        uploaded = files.upload()
        if uploaded:
            name = list(uploaded.keys())[0]
            WORK_REF_AUDIO.write_bytes(uploaded[name])
            print(f"  uploaded: {WORK_REF_AUDIO}  ({WORK_REF_AUDIO.stat().st_size/1024:.1f} KB)")
    except ImportError:
        pass

if not WORK_REF_AUDIO.exists():
    # try Kaggle's file uploader
    try:
        from kaggle_secrets import UserSecretsClient  # noqa: F401
        from ipywidgets import FileUpload
        print("📤 Upload your reference audio:")
        uploader = FileUpload(accept=".wav", multiple=False)
        display(uploader)
        # Note: would need to wait for upload interactively
    except ImportError:
        pass

assert WORK_REF_AUDIO.exists(), \
    "Could not get a reference audio. Upload one or check your internet connection."
REF_AUDIO = str(WORK_REF_AUDIO)

# Resample to 24 kHz mono and save a clean working copy
wav, sr = torchaudio.load(REF_AUDIO)
if wav.shape[0] > 1:
    wav = wav.mean(0, keepdim=True)
if sr != SAMPLE_RATE:
    wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
REF_AUDIO_24K = Path("ref_audio_24k.wav")
torchaudio.save(str(REF_AUDIO_24K), wav, SAMPLE_RATE)
print(f"\n✓ ref audio: {REF_AUDIO_24K}  ({wav.shape[-1]/SAMPLE_RATE:.2f}s @ {SAMPLE_RATE} Hz)")
print(f"  source    : {REF_AUDIO}")
print(f"  ref text  : {REF_TEXT!r}")
display(Audio(str(REF_AUDIO_24K), rate=SAMPLE_RATE))

## 4. Download fine-tuned checkpoints from HuggingFace

Run once. Caches to `./ckpts/` so subsequent runs are offline.

In [ ]:
from huggingface_hub import hf_hub_download

HF_FILES = {
    "f5_indo":    ("f5_indo/f5_indo.pt",       "f5_indo/vocab.txt"),
    "f5_indocom": ("f5_indocom/f5_indocom.pt", "f5_indocom/vocab_com.txt"),
}

paths = {}
for tag, (remote_ckpt, remote_vocab) in HF_FILES.items():
    target = CKPTS_DIR / tag
    target.mkdir(parents=True, exist_ok=True)
    ckpt_name  = Path(remote_ckpt).name
    vocab_name = Path(remote_vocab).name
    ckpt_path  = target / ckpt_name
    vocab_path = target / vocab_name

    if not ckpt_path.exists():
        print(f"⏳ downloading {tag} checkpoint ({ckpt_name}) ...")
        t0 = time.time()
        with tempfile.TemporaryDirectory() as tmp:
            p = hf_hub_download(repo_id=HF_REPO_ID, filename=remote_ckpt,
                                cache_dir=tmp, local_dir=tmp)
            shutil.copy(p, ckpt_path)
        print(f"   ✓ done in {time.time()-t0:.1f}s  → {ckpt_path}")
    if not vocab_path.exists():
        print(f"⏳ downloading {tag} vocab ...")
        with tempfile.TemporaryDirectory() as tmp:
            p = hf_hub_download(repo_id=HF_REPO_ID, filename=remote_vocab,
                                cache_dir=tmp, local_dir=tmp)
            shutil.copy(p, vocab_path)

    paths[tag] = {"ckpt": str(ckpt_path), "vocab": str(vocab_path)}
    print(f"  {tag:11s}  ckpt={ckpt_path}  ({ckpt_path.stat().st_size/1024/1024:.1f} MB)")
print("\n✓ all fine-tuned checkpoints ready")

## 5. Shared helpers (used by all 3 models)

In [ ]:
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process

vocoder = load_vocoder(is_local=False, device=device)
print(f"✓ vocoder loaded on {device}")

def play(wav, sr=SAMPLE_RATE, label=""):
    """Display an audio player in the notebook."""
    if isinstance(wav, torch.Tensor):
        wav = wav.detach().cpu().numpy()
    wav = np.asarray(wav).squeeze()
    if label:
        print(f"▶ {label}")
    display(Audio(wav, rate=sr))

def save_wav(name, wav, sr=SAMPLE_RATE):
    """Save to outputs/ and return the path."""
    if isinstance(wav, np.ndarray):
        wav = torch.from_numpy(wav).float()
    if wav.ndim == 1:
        wav = wav.unsqueeze(0)
    p = OUTPUT_DIR / f"{name}.wav"
    torchaudio.save(str(p), wav, sr)
    return str(p)

print("✓ helpers ready")

## 6. Pick which model to load

Each model is in its own cell. Run **only the cells for the model(s) you want to use** — this saves GPU memory.

Recommended: start with `f5_indo` (1.35 GB) for a quick test, then try `f5_indocom` (5.4 GB) on Colab Pro or Kaggle if you have more VRAM.

---

### 6A. Base F5-TTS (English, ~1.2 GB, auto-downloads from `SWivid/F5-TTS`)

In [ ]:
from f5_tts.api import F5TTS

f5_base = F5TTS(device=device)
print("✓ base F5-TTS loaded")

def synth_base(text):
    wav, sr, _ = f5_base.infer(
        ref_file=str(REF_AUDIO_24K),
        ref_text=REF_TEXT,
        gen_text=text,
    )
    return wav, sr

### 6B. f5_indo (Indonesian, 1.35 GB)

In [ ]:
f5_indo_model = load_model(
    DiT, MODEL_CFG, paths["f5_indo"]["ckpt"],
    mel_spec_type=MEL_SPEC,
    vocab_file=paths["f5_indo"]["vocab"],
    use_ema=True, device=device,
)
f5_indo_model.eval()
print("✓ f5_indo loaded")

def synth_f5_indo(text):
    with torch.no_grad():
        audio, sr, _ = infer_process(
            ref_audio=str(REF_AUDIO_24K),
            ref_text=REF_TEXT,
            gen_text=text,
            model_obj=f5_indo_model,
            vocoder=vocoder,
            mel_spec_type=MEL_SPEC,
            show_info=lambda *a, **k: None,
            device=device,
        )
    return audio, sr

### 6C. f5_indocom (Indonesian combined, 5.4 GB — needs ≥8 GB VRAM)

In [ ]:
f5_indocom_model = load_model(
    DiT, MODEL_CFG, paths["f5_indocom"]["ckpt"],
    mel_spec_type=MEL_SPEC,
    vocab_file=paths["f5_indocom"]["vocab"],
    use_ema=True, device=device,
)
f5_indocom_model.eval()
print("✓ f5_indocom loaded")

def synth_f5_indocom(text):
    with torch.no_grad():
        audio, sr, _ = infer_process(
            ref_audio=str(REF_AUDIO_24K),
            ref_text=REF_TEXT,
            gen_text=text,
            model_obj=f5_indocom_model,
            vocoder=vocoder,
            mel_spec_type=MEL_SPEC,
            show_info=lambda *a, **k: None,
            device=device,
        )
    return audio, sr

## 7. Quick test — single sentence through one model

Run this if you only loaded one of the model cells above. Change the `synth_*` call to match which model you loaded.

In [ ]:
# 👇 change this to whichever model you loaded:
# synth = synth_base         # for the base English model
# synth = synth_f5_indo      # for f5_indo
# synth = synth_f5_indocom   # for f5_indocom
synth = synth_f5_indo

text = "Tolong bantu aku untuk mengisi kuisioner ini yah."
wav, sr = synth(text)
play(wav, sr, label=text)

## 8. A/B/C compare — all three models on the same sentence

Only works if you ran cells 6A, 6B, **and** 6C. Otherwise skip the missing ones.

In [ ]:
def compare(text):
    print(f"\n{'='*70}\n  {text}\n{'='*70}")
    # Check once at function entry — `globals()` sees module-level names
    # from inside a function, even from a nested list comprehension.
    # (`dir()` would NOT — list comprehensions have their own scope.)
    has_base    = 'synth_base'       in globals()
    has_indo    = 'synth_f5_indo'    in globals()
    has_indocom = 'synth_f5_indocom' in globals()
    results = []
    for tag, fn in [("base",       synth_base       if has_base    else None),
                     ("f5_indo",    synth_f5_indo    if has_indo    else None),
                     ("f5_indocom", synth_f5_indocom if has_indocom else None)]:
        if fn is None:
            print(f"\n  [{tag}] not loaded — run its cell in section 6 first")
            continue
        t0 = time.time()
        try:
            wav, sr = fn(text)
            results.append((tag, wav, sr, time.time()-t0))
            print(f"\n▶ {tag}  ({time.time()-t0:.1f}s)")
            play(wav, sr)
        except Exception as e:
            print(f"\n✗ {tag} failed: {e}")
    return results

text = "Halo, perkenalkan nama saya Sarah dari Jakarta."
results = compare(text)

## 9. Batch test — multiple sentences through all loaded models

In [ ]:
sentences = [
    "Tolong bantu aku untuk mengisi kuisioner ini yah.",
    "Berita terbaru: pasar modal mengalami kenaikan signifikan hari ini.",
    "Jangan norak, kamu kira gampang?",
    "Mereka berangkat ke sekolah bersama-sama setiap pagi.",
    "Halo, nama saya Daniel, saya berasal dari Surabaya.",
]

for s in sentences:
    compare(s)

## 10. Interactive inference — type and listen

Edit `text` below and re-run this cell. It will run through all loaded models.

In [ ]:
text = "Saya ingin memesan dua tiket pesawat ke Bali untuk akhir pekan ini."
compare(text)

## 11. Save outputs to disk

All generated audio lands in `outputs/`. You can download the folder from Colab's file browser (left sidebar).

In [ ]:
def safe(s):
    return "".join(c if c.isalnum() else "_" for c in s)[:50]

# Use globals() — safer than dir() inside comprehensions / nested scopes.
g = globals()
loaded = []
if 'synth_base'       in g: loaded.append(("base",       synth_base))
if 'synth_f5_indo'    in g: loaded.append(("f5_indo",    synth_f5_indo))
if 'synth_f5_indocom' in g: loaded.append(("f5_indocom", synth_f5_indocom))

if not loaded:
    print("⚠ no models loaded — run cells in section 6 first")
else:
    print(f"  saving through: {[t for t,_ in loaded]}")
    for s in sentences:
        for tag, fn in loaded:
            try:
                wav, sr = fn(s)
                p = save_wav(f"{tag}_{safe(s)}", wav, sr)
                print(f"  saved : {p}")
            except Exception as e:
                print(f"  failed: {tag} on {s!r}: {e}")

    print(f"\n✓ all outputs in {OUTPUT_DIR}/")
    print("  download from Colab: click the folder icon in the left sidebar")

## 12. Free GPU memory

Run this when you're done to release the models.

In [ ]:
import gc
g = globals()
for name in ["f5_base", "f5_indo_model", "f5_indocom_model"]:
    if name in g:
        del g[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✓ GPU memory freed")